In [ ]:
from mindvision.dataset import Mnist
import mindspore.nn as nn
import mindspore as ms
from mindvision.engine.callback import LossMonitor
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 数据集准备
download_train = Mnist(
    path="./mnist",
    split="train",
    batch_size=32,
    repeat_num=1,
    shuffle=True,
    resize=32,
    download=True)
download_eval = Mnist(
    path="./mnist",
    split="test",
    batch_size=32,
    resize=32,
    download=True)
dataset_train = download_train.run()
dataset_eval = download_eval.run()

In [ ]:
len(dataset_train), len(dataset_eval)

In [ ]:
# 模型定义
class LeNet5(nn.Cell):
    def __init__(self, num_class=10, num_channel=1):
        super(LeNet5, self).__init__()
        # 卷积层，输入的通道数为num_channel，输出的通道数为6，卷积核大小为5*5
        self.conv1 = nn.Conv2d(num_channel, 6, 5, pad_mode='valid')
        # 卷积层，输入的通道数为6，输出的通道数为16，卷积核大小为5*5
        self.conv2 = nn.Conv2d(6, 16, 5, pad_mode='valid')
        # 全连接层，输入个数为16*5*5，输出个数为120
        self.fc1 = nn.Dense(16 * 5 * 5, 120)
        # 全连接层，输入个数为120，输出个数为84
        self.fc2 = nn.Dense(120, 84)
        # 全连接层，输入个数为84，分类的个数为num_class
        self.fc3 = nn.Dense(84, num_class)
        # ReLU激活函数
        self.relu = nn.ReLU()
        # 池化层
        self.max_pool2d = nn.MaxPool2d(kernel_size=2, stride=2)
        # 多维数组展平为一维数组
        self.flatten = nn.Flatten()

    def construct(self, x):
        # 使用定义好的运算构建前向网络
        x = self.conv1(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

In [ ]:
# 模型实例化
net = LeNet5(num_class=10)

In [ ]:
# 损失函数：交叉熵损失
loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction='mean')
# 优化器
optimizer = nn.Momentum(net.trainable_params(), learning_rate=0.001, momentum=0.9)

In [ ]:
# MindSpore需要提前声明网络模型在训练过程中是否需要保存中间过程和结果
# 设置模型保存参数
config_ck = ms.CheckpointConfig(save_checkpoint_steps=1875, keep_checkpoint_max=10)
# 应用模型保存参数
checkpoint = ms.ModelCheckpoint(prefix="lenet", directory="./lenet", config=config_ck)

In [ ]:
# 模型训练初始化
model = ms.Model(network=net,
                 loss_fn=loss,
                 optimizer=optimizer,
                 metrics={'accuracy'})

In [ ]:
# 训练网络模型
model.train(100, dataset_train, callbacks=[checkpoint, LossMonitor(0.01, 1875)])

In [ ]:
# 在测试集上进行模型的表现测试

acc = model.eval(dataset_eval)
print("{}".format(acc))

In [ ]:
# 加载模型的参数
param_dict = ms.load_checkpoint("./lenet/lenet-10_1875.ckpt")
# 加载参数到网络中
ms.load_param_into_net(net, param_dict)

In [ ]:
mnist = Mnist("./mnist", split="train", batch_size=9, resize=32)
dataset_infer = mnist.run()
ds_test = dataset_infer.create_dict_iterator()
data = next(ds_test)
images = data["image"].asnumpy()
labels = data["label"].asnumpy()

plt.figure()
for i in range(1, 10):
    plt.subplot(3, 3, i)
    plt.imshow(images[i-1][0], interpolation="None", cmap="gray")
plt.show()

# 使用函数model.predict预测image对应分类
output = model.predict(ms.Tensor(data['image']))
predicted = np.argmax(output.asnumpy(), axis=1)

# 输出预测分类与实际分类
print(f'Predicted: "{predicted}", Actual: "{labels}"')